In [ ]:
CHALLENGE 1

# Gemini Weather Agent

This project demonstrates how to build a conversational AI agent using the Google Agent Development Kit (ADK) and Vertex AI's Gemini models. The agent is capable of fetching real-time weather data for any given city using an external API.

## Features

- **Real-time Weather Data**: Uses the OpenWeatherMap API to get current weather conditions.
- **Tool-Using Agent**: The Gemini model is equipped with a `get_weather` tool that it can decide to call based on the user's query.
- **Asynchronous Testing**: Includes a test suite to verify the agent's functionality across multiple cities.
- **Extensible**: Built with a clean structure that is easy to extend with new tools and capabilities.

## Prerequisites

Before you begin, you will need:

- Python 3.10+
- A Google Cloud Project with the Vertex AI API enabled.
- An API key from OpenWeatherMap.

## Project Setup

Follow these steps to set up and run the project on your local machine.

### 1. Create the Project Structure

Create the following directory and file structure for your project:

```
weather-agent/
├── app/
│   ├── __init__.py
│   ├── agent.py
│   ├── agent_engine_app.py
│   └── test_agent.py
└── .env
```

### 2. Install Dependencies

Create a `requirements.txt` file with the following libraries:

```txt
google-cloud-aiplatform
google-adk
requests
python-dotenv
```

Install them using pip:

```bash
pip install -r requirements.txt
```

### 3. Configure Environment Variables

Create a `.env` file in the root of your `weather-agent` directory and add your project-specific keys and IDs. The `load_dotenv()` function in `agent_engine_app.py` will automatically load these at runtime.

```env
# .env
GOOGLE_CLOUD_PROJECT="your-gcp-project-id"
GOOGLE_CLOUD_LOCATION="us-central1"
WEATHER_API_KEY="your-openweathermap-api-key"
# Optional: For storing artifacts in GCS when deployed
LOGS_BUCKET_NAME="your-gcs-bucket-name"
```

### 4. Write the Agent Code

Populate the files in the `app/` directory with the code provided in the other sections of this project.

- **`app/agent.py`**: This is the core of the project. It defines the `get_weather` and `get_current_time` tools and configures the `Agent` with the Gemini model, instructions, and tools.
- **`app/__init__.py`**: This file makes the `app` directory a Python package and exports the `app` object from `agent.py`.
- **`app/test_agent.py`**: This script contains the asynchronous test function `test_weather_for_cities` that invokes the agent for a list of cities and prints the results.
- **`app/agent_engine_app.py`**: This file sets up the application for deployment on Vertex AI Agent Engine. It handles initialization, logging, and telemetry.

## How to Run the Test

Once all the files are in place and the dependencies are installed, you can test the agent from the root of the `weather-agent` directory.

The test script will:
1. Initialize the Vertex AI SDK.
2. Create an `AdkApp` instance to interact with your agent.
3. Loop through a list of cities and send a weather query for each one.
4. Print the agent's response to the console.

Run the following command:

```bash
python3 -m app.test_agent
```

You should see output showing the weather query for each city followed by the agent's response, which is fetched from the OpenWeatherMap API.

This `README.md` should give anyone a clear path to understanding and recreating your project.

CHALLENGE 2

# Extending Your Agent with Callbacks

This document outlines how to enhance the functionality of your agent by implementing callback functions. Callbacks allow you to hook into the agent's execution lifecycle to perform tasks such as logging, input validation, and more.

We will cover the following:
1.  **Logging User Prompts and Model Responses:** Keep a record of the conversation.
2.  **Validating User Input:** Ensure inputs are safe and meet specific criteria before being processed by the model.

## 1. Logging Callbacks

Logging is essential for debugging, monitoring, and understanding how your agent is being used. We'll create a callback handler to log user prompts and the model's final responses.

For this, you can use a framework like LangChain, which provides a `BaseCallbackHandler` that you can extend.

### Implementation

Here is an example of a custom callback handler for logging:

```python
from langchain.callbacks.base import BaseCallbackHandler
from langchain.schema import LLMResult
from typing import Any, Dict, List

class LoggingCallbackHandler(BaseCallbackHandler):
    """Callback Handler that logs user prompts and model responses."""

    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> None:
        """Log the user's prompt."""
        # The 'prompts' list contains the input to the language model.
        user_prompt = prompts
        print(f"\n----- User Prompt -----\n{user_prompt}")

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        """Log the model's response."""
        # The 'response' object contains the output from the language model.
        model_response = response.generations.text
        print(f"\n----- Model Response -----\n{model_response}")

# To use this callback, you would add it to your agent's execution call.
# For example, with a LangChain LLM:
#
# from langchain_openai import ChatOpenAI
#
# llm = ChatOpenAI(callbacks=[LoggingCallbackHandler()])
# llm.invoke("Tell me a fun fact about the Roman Empire.")
```

This handler taps into two key events:
*   `on_llm_start`: This is triggered right before the language model is called, giving you access to the prompts.
*   `on_llm_end`: This is triggered after the language model returns a response.

## 2. Input Validation Callback

Input validation is a critical security measure. It helps prevent malicious inputs and ensures that the data sent to your agent is in the correct format. We will create a callback that performs two checks:
*   Verifies the user's location is within the United States.
*   Scans for potentially malicious input.

### a. Location Validation

The NWS API, which this agent uses, only supports locations within the United States. We can use the `usaddress` library to parse the user's input and check if it's a valid US address.

First, install the necessary library:
```bash
pip install usaddress
```

### b. Malicious Input Detection

To guard against prompt injection attacks, we can use a library like `pytector`. This library helps in identifying prompts that are designed to manipulate the language model.

First, install the library:
```bash
pip install pytector
```

### Implementation

Here is an example of a callback handler that validates user input:

```python
import usaddress
from pytector import PromptInjectionDetector
from langchain.callbacks.base import BaseCallbackHandler
from typing import Any, Dict, List

class InputValidationCallbackHandler(BaseCallbackHandler):
    """Callback Handler that validates user input for location and malicious content."""

    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> None:
        """Validate the user's input at the start of the chain."""
        user_input = inputs.get("input", "")

        if not user_input:
            return

        print("\n----- Running Input Validation -----")

        # 1. Location Validation
        try:
            # The usaddress library can parse unstructured US addresses.
            # If it successfully parses the address and identifies address components,
            # we can be reasonably sure it's a US location.
            tagged_address, address_type = usaddress.tag(user_input)
            if not any(key in ['PlaceName', 'StateName', 'ZipCode'] for key in tagged_address):
                 raise ValueError("Location appears to be outside of the United States.")
            print("Location validation successful.")
        except Exception as e:
            raise ValueError(f"Location validation failed: {e}")

        # 2. Malicious Input Detection
        detector = PromptInjectionDetector()
        detection_result = detector.detect_injection(user_input)
        is_injection, probability = detection_result

        if is_injection:
            raise ValueError(f"Malicious input detected with probability: {probability}")
        else:
            print("Malicious input check successful.")
        
        print("------------------------------------")
```

By implementing these callbacks, you can significantly improve the robustness, security, and observability of your agent.

**CHALLENGE 3**

# Building an Agent Team for Verified and Refined Answers

This document outlines how to create a sophisticated workflow using multiple agents to answer questions, where responses are verified and refined before being returned to the user. This pattern enhances the quality and reliability of your agent's output.

We will use a workflow with an "answer team" composed of multiple specialized agents:
1.  **Search Agent**: Its job is to gather the initial information to answer a user's question.
2.  **Critique Agent**: This agent reviews the initial answer and provides constructive feedback for improvement.
3.  **Refine Agent**: This agent takes the original answer and the critique to produce a final, polished response.

These agents will be orchestrated using a `SequentialAgent`, which runs them in a defined order.

## 1. Defining the Agent Team

First, we define each of our specialized agents. Each agent is an `LlmAgent` with a specific instruction, and in the case of the Search agent, a specific tool.

### a. Search Agent

This agent's role is to find the data needed to answer the question. We'll give it a `google_search` tool to find up-to-date information.

```python
from google.adk.agents import LlmAgent
from google.adk.tools import google_search

search_agent = LlmAgent(
    model="gemini-1.5-flash",
    instruction="""
    You are a search agent. Your job is to find the information to answer the user's question.
    Use the search tool to find the most relevant and up-to-date information.
    Present the facts clearly.
    """,
    tools=[google_search],
)
```

### b. Critique Agent

This agent acts as a reviewer. It takes the output from the `search_agent` and identifies any potential issues, such as inaccuracies, lack of detail, or poor formatting.

```python
critique_agent = LlmAgent(
    model="gemini-1.5-flash",
    instruction="""
    You are a critique agent. Your role is to review the provided answer for quality.
    Look for issues with accuracy, clarity, and completeness.
    Provide specific, constructive feedback on how the answer can be improved.
    """,
)
```

### c. Refine Agent

The final agent in our sequence takes the original answer and the critique and rewrites it into a better response.

```python
refine_agent = LlmAgent(
    model="gemini-1.5-flash",
    instruction="""
    You are a refine agent. Your job is to rewrite the original answer based on the critique provided.
    Incorporate the feedback to create a more accurate, clear, and complete response.
    Present the final answer to the user.
    """,
)
```

## 2. Orchestrating the Workflow with a Sequential Agent

Now that we have our team of agents, we can use a `SequentialAgent` to manage the workflow. The `SequentialAgent` will execute our agents in the order we define: `search_agent`, then `critique_agent`, and finally `refine_agent`. The output of each agent is passed as input to the next.

```python
from google.adk.agents import SequentialAgent

answer_team = SequentialAgent(
    sub_agents=[search_agent, critique_agent, refine_agent]
)
```

## 3. Testing Your Agent Team

To test this multi-agent system, you can create an `AdkApp` and stream the query. By listening for `step_end` events, you can observe the output from each sub-agent as it executes.

Here is a sample test script:

```python
import asyncio
import vertexai
from vertexai.agent_engines.templates.adk import AdkApp

# Assuming 'answer_team' is the SequentialAgent defined above
# and it's part of an ADK App instance like this:
# from google.adk.apps import App
# app = App(root_agent=answer_team)

async def test_answer_team():
    """Tests the multi-agent answer team."""
    vertexai.init(project="your-gcp-project-id")
    agent_engine_app = AdkApp(app=app) # 'app' is your ADK App instance

    query = "What were the key findings of the latest IPCC report on climate change?"
    response_generator = agent_engine_app.async_stream_query(
        message=query, user_id="test_user"
    )

    print(f"Query: {query}\n")
    final_response = ""
    async for event in response_generator:
        if event["type"] == "step_end":
            print(f"--- Output from {event['agent_name']} ---")
            print(event["outputs"]["llm_response"])
            print("-" * (len(event['agent_name']) + 14))
        elif event["type"] == "text":
            final_response += event["text"]

    print("\n--- Final Refined Response ---")
    print(final_response)

if __name__ == "__main__":
    # You would need to place the agent definitions and app creation
    # in scope for this to run.
    # asyncio.run(test_answer_team())
    print("Test script structure is ready.")
```

When you run this test, the output will show the step-by-step process: the initial search results, the critique of those results, and the final, refined answer that is presented to the user. This demonstrates the power of using a multi-agent workflow to improve response quality.

**CHALLENGE 4**

# Building a Multi-Agent System with a Root Coordinator

This document explains how to construct a multi-agent system where a primary "root" agent delegates tasks to specialized sub-agents. This architecture is powerful for creating more capable and organized AI systems.

We will build a system with three agents:
1.  **Weather Agent**: A specialized agent that can fetch real-time weather data.
2.  **Search Agent**: A specialized agent that can answer general knowledge questions using Google Search.
3.  **Root Agent**: The central coordinator that receives all user requests and decides which sub-agent is best suited to handle the task.

## 1. Defining the Sub-Agents

First, we'll create our specialized sub-agents. Each will have a specific instruction and a tool tailored to its purpose.

### a. The Weather Agent

This agent is responsible for all weather-related queries. It uses the `get_weather` tool defined in `app/agent.py`.

```python
from google.adk.agents import Agent
from google.adk.models import Gemini

# Assume get_weather is defined as in app/agent.py

weather_agent = Agent(
    name="weather_agent",
    model=Gemini(model="gemini-1.5-flash"),
    instruction="You are a weather agent. Your job is to use the get_weather tool to find the weather for a given city.",
    tools=[get_weather],
)
```

### b. The Search Agent

This agent handles general knowledge questions by using the ADK's built-in `google_search` tool.

```python
from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.tools import google_search

search_agent = Agent(
    name="search_agent",
    model=Gemini(model="gemini-1.5-flash"),
    instruction="You are a search agent. Your job is to use the google_search tool to answer the user's question.",
    tools=[google_search],
)
```

## 2. Defining the Root Agent

The `root_agent` acts as the entry point and orchestrator. It doesn't have tools of its own; instead, its instruction tells it how to delegate tasks to its sub-agents based on the user's prompt.

We register the `weather_agent` and `search_agent` in the `sub_agents` list.

```python
root_agent = Agent(
    name="root_agent",
    model=Gemini(model="gemini-1.5-flash"),
    instruction="""You are a helpful AI assistant. Your goal is to delegate tasks to the appropriate sub-agent.

    If the user asks about the weather, delegate the task to the 'weather_agent'.
    For any other general question, delegate the task to the 'search_agent'.
    """,
    sub_agents=[weather_agent, search_agent],
)
```

Finally, we wire up the `root_agent` to our application.

```python
from google.adk.apps import App

app = App(
    root_agent=root_agent,
    name="app",
)
```

## 3. Testing the Multi-Agent System

To see the delegation in action, we can send different queries to the `root_agent` and observe which sub-agent gets activated. The following test script demonstrates this by checking the `step_start` event, which contains the name of the agent beginning a turn.

```python
import asyncio
import vertexai
from vertexai.preview.autologging import Autologging
from vertexai.agent_engines.templates.adk import AdkApp

# Assuming 'app' is the ADK App instance defined above

async def test_delegation():
    """Tests the root agent's ability to delegate to sub-agents."""
    vertexai.init(project="your-gcp-project-id")
    Autologging.init()
    agent_engine_app = AdkApp(app=app)

    queries = [
        "What is the weather like in London?",
        "Who was the first person to walk on the moon?",
    ]

    for query in queries:
        print(f"--- Testing Query: '{query}' ---")
        response_generator = agent_engine_app.async_stream_query(
            message=query, user_id="test_user"
        )

        final_response = ""
        async for event in response_generator:
            if event["type"] == "step_start":
                # This event shows which agent is starting its turn.
                agent_name = event.get("agent_name")
                if agent_name != "root_agent":
                    print(f"  >> Delegated to: {agent_name}")
            
            elif event["type"] == "text":
                final_response += event["text"]

        print(f"Final Response: {final_response}\n")


if __name__ == "__main__":
    # You would need to place the agent definitions and app creation
    # in scope for this to run.
    # asyncio.run(test_delegation())
    print("Test script structure is ready.")

```

When you run this test, you will see output confirming that the weather query is delegated to the `weather_agent` and the general knowledge query is delegated to the `search_agent`, demonstrating a successful multi-agent implementation.

**CHALLENGE 5**

# Deploying Your Agent to Vertex AI Agent Engine

This guide provides a step-by-step process for deploying your ADK-based agent to Vertex AI Agent Engine. Deployment allows your agent to be used as a scalable, managed service.

We will cover the following steps:
1.  Initialize Vertex AI
2.  Define Your Agent
3.  Test Your Agent Locally
4.  Deploy and Manage Your Agent with Agent Engine

## 1. Initialize Vertex AI

Before you can deploy, you need to initialize the Vertex AI SDK with your Google Cloud project and location. This tells the SDK where to deploy your resources.

```python
import vertexai

# Replace with your project and location
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
```

## 2. Define Your Agent

Your agent should be defined using the Google Agent Development Kit (ADK). This involves creating an `Agent` and wrapping it in an `App`. The following is a basic structure, similar to what you would have in your `app/agent.py` file.

```python
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.models import Gemini
from google.adk.tools import google_search

# Define a simple agent
my_agent = Agent(
    name="my_agent",
    model=Gemini(model="gemini-1.5-flash"),
    instruction="You are a helpful assistant.",
    tools=[google_search],
)

# Wrap the agent in an ADK App
app = App(
    root_agent=my_agent,
    name="my-awesome-app",
)
```

## 3. Test Your Agent Locally

Before deploying, it's crucial to test your agent in your local environment to ensure it behaves as expected. You can use the `AdkApp` class from the ADK templates to simulate the agent's execution.

```python
import asyncio
from vertexai.agent_engines.templates.adk import AdkApp

async def local_test():
    """Tests the agent locally before deployment."""
    agent_engine_app = AdkApp(app=app) # 'app' is your ADK App instance

    query = "What is the capital of France?"
    response_generator = agent_engine_app.async_stream_query(
        message=query, user_id="local_test_user"
    )

    print(f"Query: {query}\n")
    final_response = ""
    async for event in response_generator:
        if event["type"] == "text":
            final_response += event["text"]

    print(f"Final Response: {final_response}")

# To run the test:
# asyncio.run(local_test())
```

## 4. Deploy and Manage Your Agent

Once you have tested your agent, you can deploy it using the `agent_engines.create()` method. This will create a long-running, managed version of your agent.

```python
from vertexai.preview import agent_engines

# 'app' is your ADK App instance from app/agent.py

# Deploy the agent
deployed_agent = agent_engines.create(
    app=app,
    display_name="My Deployed Weather Agent",
)

print(f"Agent deployed successfully. Resource Name: {deployed_agent.resource_name}")

# Start a new session with the deployed agent
session = deployed_agent.new_session()

# Send a request to the deployed agent
response = session.query("What's the weather in New York?")
print(f"Response from deployed agent: {response}")
```

By following these steps, you can take your locally-defined agent and turn it into a robust, scalable service on Vertex AI.

**CASE STUDY**

# FEMA Agent Readme

The FEMA Agent can provide weather forecasts, search the internet, provide routes using the Google Maps API, and answer questions. It uses a sequential workflow that validates and refines responses.

This document provides comprehensive instructions on how to set up, run, and deploy the FEMA Agent.

## Prerequisites

Before you begin, ensure you have the following prerequisites:

*   A Google Cloud project with the Vertex AI API enabled.
*   The Google Cloud SDK (gcloud) installed and configured.
*   Python 3.7 or later installed.
*   Poetry for dependency management.

## Setup Instructions

1.  **Clone the repository:**
    git clone https://github.com/GoogleCloudPlatform/agent-starter-pack
    cd agent-starter-pack
    ```

2.  **Clone the agent repo:**
    ```bash
    git clone <repository_url>
    cd weatheragent2
    ```

2.  **Configure Google Cloud:**

    Set your Google Cloud project ID:

    ```bash
    gcloud config set project qwiklabs-gcp-01-0c244accce49
    ```

3.  **Install dependencies:**

    ```bash
    poetry install
    ```

## Running the Agent Locally

To run the agent locally, execute the following command:

```bash
python3 /home/student_02_4947adaf7321/weatheragent2/test_fema_agent.py
```

This command will run the test script, which initializes the agent and sends a query to it. The agent's response will be printed to the console.

## Deploying the Agent to Agent Engine

To deploy the agent to Agent Engine, execute the following command:

```bash
python3 /home/student_02_4947adaf7321/weatheragent2/deploy_fema_agent.py
```

This command will deploy the agent to Agent Engine. The output will include the resource name and display name of the deployed agent.